In [0]:

























































































































%pip install lightgbm shap seaborn --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import os

#  fontes (Camada Silver)
SILVER_BASE = "/Volumes/workspace/olist_silver/parquet"
TABELAS = {
    "customers": "olist_customers_dataset",
    "orders": "olist_orders_dataset",
    "items": "olist_order_items_dataset",
    "payments": "olist_order_payments_dataset",
    "reviews": "olist_order_reviews_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "translation": "product_category_name_translation",
}

TABELA_GEO_OPCIONAL = "olist_geolocation_dataset"


SAIDA_DIR = "/Volumes/workspace/olist_silver/parquet/_ml_saidas"  

# 
DATA_INICIO, DATA_FIM = "2017-01-01", "2018-08-31"  
FOLDS_VALIDACAO = [                                   
    ("2017-07-01", "2017-10-01"),
    ("2017-10-01", "2018-01-01"),
    ("2018-01-01", "2018-04-01"),
]
FIM_TREINO_FINAL = "2018-01-01"   # modelo final: treina antes daqui ...
INICIO_TESTE = "2018-04-01"       # ... calibra em jan-mar/2018 e é testado uma vez em abr-ago/2018
EMBARGO_DIAS = 30                 # pedidos comprados nos 30 dias antes de um corte não entram no treino
CALIBRAR = True                   # calibrador sigmoide ajustado em jan-mar/2018 (o Brier sem calibração é sempre reportado)

# features históricas
JANELA_RECENTE_DIAS = 30
MIN_EVENTOS_ROTA = 30             # rota UF>UF precisa de 30+ entregas anteriores; senão cai para região>região
MIN_EVENTOS_RECENTE = 20
MIN_EVENTOS_VENDEDOR = 3

# busca e reprodutibilidade
SEED = 42
N_ITER_BUSCA = 10                
MODO_RAPIDO = False               
if MODO_RAPIDO:
    N_ITER_BUSCA = 4

#  matriz de decisão (definida a priori)
PESOS_DECISAO = {
    "discriminacao_pr_auc": 0.35,     # classes desbalanceadas: PR-AUC e não acurácia
    "calibracao_brier": 0.20,         # a probabilidade vai para o app, então precisa ser confiável
    "alto_risco_precisao_top10": 0.15,  # utilidade da faixa vermelha (10% de maior risco)
    "estabilidade": 0.10,             # queda validação->teste e variação mês a mês
    "interpretabilidade": 0.10,       # 1 = coeficientes diretos; 0,5 = precisa de SHAP/permutação
    "viabilidade_operacional": 0.10,  # latência e tamanho do artefato
}
assert abs(sum(PESOS_DECISAO.values()) - 1) < 1e-9

In [0]:
import io
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
PCT = mtick.PercentFormatter(1)
np.random.seed(SEED)

try:
    os.makedirs(os.path.join(SAIDA_DIR, "figuras"), exist_ok=True)
except OSError:
    SAIDA_DIR = "/tmp/olist_ml_saidas"
    os.makedirs(os.path.join(SAIDA_DIR, "figuras"), exist_ok=True)
    print(f"AVISO: não foi possível escrever no volume; saídas em {SAIDA_DIR} (temporário)")
DIR_FIG = os.path.join(SAIDA_DIR, "figuras")


def mostrar(fig, nome):
    """Mostra o gráfico no notebook e salva uma cópia em SAIDA_DIR/figuras."""
    fig.tight_layout()
    fig.savefig(os.path.join(DIR_FIG, f"{nome}.png"), dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)

 1. Leitura e integração da Camada Silver (PySpark)



In [0]:
# [spark]
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def ler_silver(nome_tabela, obrigatoria=True):
    """Lê uma tabela Parquet da Silver. Aceita pasta (<nome>/) ou arquivo (<nome>.parquet)."""
    for caminho in (f"{SILVER_BASE}/{nome_tabela}", f"{SILVER_BASE}/{nome_tabela}.parquet"):
        try:
            return spark.read.parquet(caminho)
        except Exception:
            continue
    if obrigatoria:
        raise FileNotFoundError(f"Tabela {nome_tabela} não encontrada em {SILVER_BASE}")
    return None


silver = {chave: ler_silver(nome) for chave, nome in TABELAS.items()}
silver_geo = ler_silver(TABELA_GEO_OPCIONAL, obrigatoria=False)

for chave, df in silver.items():
    print(f"{TABELAS[chave]:40s} {df.count():>9,} linhas | {len(df.columns)} colunas")
print(f"{TABELA_GEO_OPCIONAL:40s} {'disponível' if silver_geo is not None else 'ausente (distância pela capital da UF)'}")

In [0]:
# [spark]
# Normalização defensiva: corrige nomes com erro de digitação da base original, tipos e CEP.
# Se a Silver já estiver padronizada, estas operações não alteram nada.
RENOMEAR = {"products": {"product_name_lenght": "product_name_length",
                         "product_description_lenght": "product_description_length"}}
DATAS = {"orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                    "order_delivered_customer_date", "order_estimated_delivery_date"],
         "items": ["shipping_limit_date"],
         "reviews": ["review_creation_date", "review_answer_timestamp"]}
NUMERICOS = {"items": ["price", "freight_value", "order_item_id"],
             "payments": ["payment_value", "payment_installments", "payment_sequential"],
             "reviews": ["review_score"],
             "products": ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
                          "product_photos_qty", "product_description_length", "product_name_length"]}
CEPS = {"customers": "customer_zip_code_prefix", "sellers": "seller_zip_code_prefix"}
OBRIGATORIAS = {
    "orders": ["order_id", "customer_id", "order_status"] + DATAS["orders"],
    "customers": ["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"],
    "items": ["order_id", "order_item_id", "product_id", "seller_id", "shipping_limit_date", "price", "freight_value"],
    "payments": ["order_id", "payment_type", "payment_installments", "payment_value"],
    "reviews": ["order_id", "review_score"],
    "products": ["product_id", "product_category_name", "product_weight_g", "product_length_cm",
                 "product_height_cm", "product_width_cm"],
    "sellers": ["seller_id", "seller_zip_code_prefix", "seller_city", "seller_state"],
    "translation": ["product_category_name", "product_category_name_english"],
}


def normalizar(chave, df):
    for antigo, novo in RENOMEAR.get(chave, {}).items():
        if antigo in df.columns and novo not in df.columns:
            df = df.withColumnRenamed(antigo, novo)
    faltando = [c for c in OBRIGATORIAS[chave] if c not in df.columns]
    assert not faltando, f"{TABELAS[chave]}: colunas ausentes na Silver {faltando}"
    for c in DATAS.get(chave, []):
        if c in df.columns:
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS TIMESTAMP)"))
    for c in NUMERICOS.get(chave, []):
        if c in df.columns:
            df = df.withColumn(c, F.expr(f"try_cast(`{c}` AS DOUBLE)"))    # evita Decimal virar 'object' no pandas
    if chave in CEPS:
        c = CEPS[chave]
        df = df.withColumn(c, F.lpad(F.col(c).cast("int").cast("string"), 5, "0"))
    return df


silver = {chave: normalizar(chave, df) for chave, df in silver.items()}

# 1 avaliação por pedido (a mais recente). Na base original, review_id não é chave única.
col_ordem = "review_answer_timestamp" if "review_answer_timestamp" in silver["reviews"].columns else "review_creation_date"
w_rev = Window.partitionBy("order_id").orderBy(F.col(col_ordem).desc_nulls_last())
silver["reviews"] = (silver["reviews"].withColumn("_rn", F.row_number().over(w_rev))
                     .filter("_rn = 1").drop("_rn"))

# Chaves únicas esperadas: se falhar, a Silver tem duplicatas que distorceriam os joins
for chave, pk in [("orders", ["order_id"]), ("customers", ["customer_id"]), ("products", ["product_id"]),
                  ("sellers", ["seller_id"]), ("items", ["order_id", "order_item_id"]), ("reviews", ["order_id"])]:
    df = silver[chave]
    dup = df.count() - df.select(*pk).distinct().count()
    assert dup == 0, f"{TABELAS[chave]}: {dup} linhas com chave {pk} duplicada"
print("Checagem de chaves OK")

In [0]:

#  itens + produtos -> 1 linha por pedido
prod = (silver["products"]
        .join(silver["translation"], "product_category_name", "left")
        .withColumn("product_category_name", F.coalesce("product_category_name", F.lit("sem_categoria")))
        .withColumn("volume_cm3", F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
        .withColumn("maior_dim_cm", F.greatest("product_length_cm", "product_height_cm", "product_width_cm")))
for c in ["product_photos_qty", "product_description_length", "product_name_length"]:
    if c not in prod.columns:
        prod = prod.withColumn(c, F.lit(None).cast("double"))

itens = silver["items"].join(prod.select("product_id", "product_category_name", "product_category_name_english",
                                         "product_weight_g", "volume_cm3", "maior_dim_cm", "product_photos_qty",
                                         "product_description_length", "product_name_length"),
                             "product_id", "left")

agg_itens = itens.groupBy("order_id").agg(
    F.count("*").alias("n_itens"),
    F.countDistinct("product_id").alias("n_produtos_distintos"),
    F.countDistinct("seller_id").alias("n_vendedores"),
    F.sum("price").alias("valor_produtos"),
    F.sum("freight_value").alias("valor_frete"),
    F.avg("price").alias("preco_medio_item"),
    F.max("price").alias("preco_max_item"),
    F.sum("product_weight_g").alias("peso_total_g"),
    F.max("product_weight_g").alias("peso_max_g"),
    F.sum("volume_cm3").alias("volume_total_cm3"),
    F.max("maior_dim_cm").alias("maior_dimensao_cm"),
    F.avg("product_photos_qty").alias("fotos_media"),
    F.avg("product_description_length").alias("tam_descricao_media"),
    F.avg("product_name_length").alias("tam_nome_medio"),
    F.max("shipping_limit_date").alias("data_limite_postagem"),   # pós-compra (ver dicionário): só EDA
)

# Item principal (maior preço) define a categoria e o vendedor de referência do pedido
w_item = Window.partitionBy("order_id").orderBy(F.col("price").desc(), F.col("order_item_id").asc())
principal = (itens.withColumn("_rn", F.row_number().over(w_item)).filter("_rn = 1")
             .select("order_id", F.col("product_category_name").alias("categoria"),
                     F.col("product_category_name_english").alias("categoria_en"),
                     F.col("seller_id").alias("vendedor_principal")))

# ----------------------------------------------------------------- pagamentos -> 1 linha por pedido
pag = silver["payments"]
w_pag = Window.partitionBy("order_id").orderBy(F.col("payment_value").desc())
tipo_pag = (pag.withColumn("_rn", F.row_number().over(w_pag)).filter("_rn = 1")
            .select("order_id", F.col("payment_type").alias("tipo_pagamento")))
agg_pag = pag.groupBy("order_id").agg(
    F.sum("payment_value").alias("valor_pago"),
    F.countDistinct("payment_type").alias("n_formas_pagamento"),
    F.max(F.greatest(F.col("payment_installments"), F.lit(1.0))).alias("parcelas"),   # 0 parcelas = à vista
    F.sum(F.when(F.col("payment_type") == "voucher", F.col("payment_value")).otherwise(0.0)).alias("valor_voucher"),
).join(tipo_pag, "order_id", "left")

# ----------------------------------------------------------------- avaliação (pós-entrega: só EDA de satisfação)
rev = silver["reviews"]
msg = F.col("review_comment_message") if "review_comment_message" in rev.columns else F.lit(None).cast("string")
agg_rev = rev.select("order_id", "review_score",
                     F.when(F.length(F.trim(msg)) > 0, 1).otherwise(0).alias("tem_comentario"))

# ----------------------------------------------------------------- geolocalização opcional (1 ponto por CEP)
geo = None
if silver_geo is not None:
    geo = (silver_geo
           .withColumn("cep", F.lpad(F.col("geolocation_zip_code_prefix").cast("int").cast("string"), 5, "0"))
           .filter(F.col("geolocation_lat").between(-34, 5.5) & F.col("geolocation_lng").between(-74, -34))
           .groupBy("cep").agg(F.avg("geolocation_lat").alias("lat"), F.avg("geolocation_lng").alias("lng")))

# ----------------------------------------------------------------- base integrada (1 linha por pedido)
clientes = silver["customers"].select("customer_id", "customer_unique_id", "customer_zip_code_prefix",
                                      "customer_city", "customer_state")
vendedores = silver["sellers"].select(F.col("seller_id").alias("vendedor_principal"), "seller_zip_code_prefix",
                                      "seller_city", "seller_state")
integrada = (silver["orders"]
             .join(clientes, "customer_id", "left")
             .join(agg_itens, "order_id", "inner")          # pedidos sem item não têm produto nem frete
             .join(principal, "order_id", "left")
             .join(vendedores, "vendedor_principal", "left")
             .join(agg_pag, "order_id", "left")
             .join(agg_rev, "order_id", "left"))
if geo is not None:
    integrada = (integrada
                 .join(geo.select(F.col("cep").alias("customer_zip_code_prefix"), F.col("lat").alias("cliente_lat"),
                                  F.col("lng").alias("cliente_lng")), "customer_zip_code_prefix", "left")
                 .join(geo.select(F.col("cep").alias("seller_zip_code_prefix"), F.col("lat").alias("vendedor_lat"),
                                  F.col("lng").alias("vendedor_lng")), "seller_zip_code_prefix", "left"))
else:
    for c in ["cliente_lat", "cliente_lng", "vendedor_lat", "vendedor_lng"]:
        integrada = integrada.withColumn(c, F.lit(None).cast("double"))

# Testes de integridade da integração
n_ped, n_unicos = integrada.count(), integrada.select("order_id").distinct().count()
assert n_ped == n_unicos, "o join duplicou pedidos"
gmv_itens = silver["items"].join(integrada.select("order_id"), "order_id", "semi").agg(F.sum("price")).first()[0]
gmv_base = integrada.agg(F.sum("valor_produtos")).first()[0]
assert abs(gmv_itens - gmv_base) < 1, "GMV da base integrada não bate com order_items"
print(f"Base integrada: {n_ped:,} pedidos x {len(integrada.columns)} colunas | GMV R$ {gmv_base:,.2f} (reconciliado)")

In [0]:
# [spark]
# A base tem ~100 mil pedidos: cabe em memória no driver. As etapas seguintes usam pandas porque as
# features históricas precisam de junções "as-of" (o que se sabia em cada instante), que o pandas resolve
# com merge_asof de forma simples e auditável. Todo o histórico (inclusive 2016) é trazido: a janela de
# análise é aplicada só depois, para que o histórico de jan/2017 não comece vazio.
pedidos = integrada.toPandas()
print(pedidos.shape)

## Base analítica e definição do alvo

**Alvo (Y):** `atrasou = 1` quando a **data** de entrega é posterior à **data** prometida. A comparação é por dia e não por horário, porque `order_estimated_delivery_date` vem à meia-noite: comparando horários, um pedido entregue às 15h do dia prometido contaria como atrasado.

**Quem entra:**
- pedidos entregues cujo prazo venceu antes do fim da base;
- pedidos postados, não cancelados e **não entregues** com prazo já vencido. São atrasos certos (alvo = 1). Descartá-los tiraria da base justamente os piores atrasos.

Pedidos com prazo ainda aberto no fim da base ficam fora, porque o resultado deles é desconhecido (censura).

**Quando o alvo fica conhecido** (`data_conhecimento_alvo`): na entrega, se chegou no prazo; no dia seguinte ao prazo, se atrasou. Essa data é usada no embargo da validação temporal.

In [0]:
REGIAO = {**dict.fromkeys(["AC", "AM", "AP", "PA", "RO", "RR", "TO"], "Norte"),
          **dict.fromkeys(["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"], "Nordeste"),
          **dict.fromkeys(["DF", "GO", "MS", "MT"], "Centro-Oeste"),
          **dict.fromkeys(["ES", "MG", "RJ", "SP"], "Sudeste"),
          **dict.fromkeys(["PR", "RS", "SC"], "Sul")}
# Capitais (nome normalizado, lat, lng): fallback da distância e feature "cliente_em_capital"
CAPITAIS = {"AC": ("rio branco", -9.97, -67.81), "AL": ("maceio", -9.67, -35.74), "AP": ("macapa", 0.03, -51.07),
            "AM": ("manaus", -3.12, -60.02), "BA": ("salvador", -12.97, -38.51), "CE": ("fortaleza", -3.73, -38.52),
            "DF": ("brasilia", -15.79, -47.88), "ES": ("vitoria", -20.32, -40.34), "GO": ("goiania", -16.68, -49.25),
            "MA": ("sao luis", -2.53, -44.30), "MT": ("cuiaba", -15.60, -56.10), "MS": ("campo grande", -20.47, -54.62),
            "MG": ("belo horizonte", -19.92, -43.94), "PA": ("belem", -1.46, -48.50), "PB": ("joao pessoa", -7.12, -34.86),
            "PR": ("curitiba", -25.43, -49.27), "PE": ("recife", -8.05, -34.88), "PI": ("teresina", -5.09, -42.80),
            "RJ": ("rio de janeiro", -22.91, -43.17), "RN": ("natal", -5.79, -35.21), "RS": ("porto alegre", -30.03, -51.23),
            "RO": ("porto velho", -8.76, -63.90), "RR": ("boa vista", 2.82, -60.67), "SC": ("florianopolis", -27.60, -48.55),
            "SP": ("sao paulo", -23.55, -46.63), "SE": ("aracaju", -10.91, -37.07), "TO": ("palmas", -10.18, -48.33)}

COLS_DATA = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date", "data_limite_postagem"]


def dias(delta):
    return delta.dt.total_seconds() / 86400


def normaliza_texto(s):
    s = s.astype("string").str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
    return s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (np.radians(np.asarray(v, dtype=float)) for v in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def preparar_base(ped):
    ped = ped.copy()
    for c in COLS_DATA:
        ped[c] = pd.to_datetime(ped[c], errors="coerce")
    ts = ped["order_purchase_timestamp"]
    entrega, prometido = ped["order_delivered_customer_date"], ped["order_estimated_delivery_date"]

    # ------------------------------------------------ variáveis PÓS-COMPRA (só EDA; nunca entram no modelo)
    ped["entregue"] = entrega.notna().astype(int)
    ped["horas_aprovacao"] = dias(ped["order_approved_at"] - ts) * 24
    ped["dias_aprovacao_postagem"] = dias(ped["order_delivered_carrier_date"] - ped["order_approved_at"])
    ped["dias_transporte"] = dias(entrega - ped["order_delivered_carrier_date"])
    ped["dias_entrega"] = dias(entrega - ts)
    ped["atraso_dias"] = (entrega.dt.normalize() - prometido.dt.normalize()).dt.days
    ped["postou_apos_limite"] = (ped["order_delivered_carrier_date"] > ped["data_limite_postagem"]).astype(float) \
        .where(ped["order_delivered_carrier_date"].notna())
    ped["review_negativa"] = (ped["review_score"] <= 2).astype(float).where(ped["review_score"].notna())

    # ------------------------------------------------ alvo
    fim_base = entrega.max().normalize()
    vencido = prometido.dt.normalize() < fim_base
    sem_entrega = ((ped["entregue"] == 0) & ped["order_delivered_carrier_date"].notna()
                   & (ped["order_status"] != "canceled") & vencido)
    ped["elegivel"] = (vencido & ((ped["entregue"] == 1) | sem_entrega)).astype(int)
    ped["atraso_sem_entrega"] = sem_entrega.astype(int)
    ped["atrasou"] = np.where(ped["entregue"] == 1, (ped["atraso_dias"] > 0).astype(float),
                              np.where(sem_entrega, 1.0, np.nan))
    dia_seguinte_prazo = prometido.dt.normalize() + pd.Timedelta(days=1)
    ped["data_conhecimento_alvo"] = entrega.where(ped["atrasou"] == 0, dia_seguinte_prazo)

    # ------------------------------------------------ atributos básicos, conhecidos na compra (usados na EDA)
    ped["prazo_prometido_dias"] = dias(prometido - ts)
    ped["ticket_total"] = ped["valor_produtos"] + ped["valor_frete"]
    ped["frete_pct_ticket"] = ped["valor_frete"] / ped["ticket_total"]
    for c in ["categoria", "tipo_pagamento", "customer_state", "seller_state"]:
        ped[c] = ped[c].fillna("desconhecido").astype(str)
    ped["regiao_cliente"] = ped["customer_state"].map(REGIAO).fillna("desconhecido")
    ped["regiao_vendedor"] = ped["seller_state"].map(REGIAO).fillna("desconhecido")
    ped["mesmo_estado"] = (ped["customer_state"] == ped["seller_state"]).astype(int)
    ped["mesma_regiao"] = (ped["regiao_cliente"] == ped["regiao_vendedor"]).astype(int)
    ped["cidade_cliente"] = normaliza_texto(ped["customer_city"])
    ped["cidade_vendedor"] = normaliza_texto(ped["seller_city"]).str.split(r"\s*[-/,\\]\s*", regex=True).str[0]

    # Distância: CEP quando há geolocalização; senão, capital da UF
    cap = pd.DataFrame(CAPITAIS, index=["nome", "lat", "lng"]).T
    for lado, uf in [("cliente", "customer_state"), ("vendedor", "seller_state")]:
        for eixo in ["lat", "lng"]:
            col = f"{lado}_{eixo}"
            ped[col] = pd.to_numeric(ped[col], errors="coerce").fillna(ped[uf].map(cap[eixo]).astype(float))
    ped["distancia_km"] = haversine_km(ped["vendedor_lat"], ped["vendedor_lng"], ped["cliente_lat"], ped["cliente_lng"])
    ped["ano_mes"] = ts.dt.to_period("M").astype(str)
    return ped


base = preparar_base(pedidos)
na_janela = base["order_purchase_timestamp"].between(DATA_INICIO, f"{DATA_FIM} 23:59:59")
base_modelo = base[na_janela & (base["elegivel"] == 1)].copy()
print(f"Pedidos na Silver integrada: {len(base):,} | na janela: {na_janela.sum():,} | elegíveis para o alvo: {len(base_modelo):,}")
print(f"Atrasos sem entrega recuperados: {base_modelo['atraso_sem_entrega'].sum():,}")
print(f"Taxa de atraso na base do modelo: {base_modelo['atrasou'].mean():.2%}")

# Análise exploratória e testes estatísticos

**Toda a EDA usa só o período de desenvolvimento (jan/2017 a mar/2018)

Com dezenas de milhares de pedidos, quase toda diferença fica "estatisticamente significativa". Por isso cada teste vem com **tamanho de efeito** (V de Cramér, correlação rank-biserial, rho de Spearman) e os p-valores são corrigidos por **Holm** para comparações múltiplas.

| Hipótese | Teste |
|---|---|
| H1. Frete mais caro (absoluto e relativo ao ticket) está associado a mais atraso | Mann-Whitney U + rank-biserial |
| H2. Distância maior aumenta o atraso | Mann-Whitney U, Spearman |
| H3. Prazo prometido mais longo reduz o atraso (folga) | Mann-Whitney U, Spearman |
| H4. UF de destino, rota entre regiões e categoria estão associadas ao atraso | Qui-quadrado + V de Cramér |
| H5. Atraso reduz a nota do cliente | Mann-Whitney U, qui-quadrado, Kruskal-Wallis, Spearman |
| H6. Preço e frete afetam a nota mesmo sem atraso | Spearman |

In [0]:
eda = base_modelo[base_modelo["order_purchase_timestamp"] < INICIO_TESTE].copy()
print(f"EDA: {len(eda):,} pedidos de {eda['order_purchase_timestamp'].min():%d/%m/%Y} a "
      f"{eda['order_purchase_timestamp'].max():%d/%m/%Y} | taxa de atraso {eda['atrasou'].mean():.2%}")


def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    ordem = np.argsort(p)
    m = len(p)
    ajust = np.maximum.accumulate((m - np.arange(m)) * p[ordem])
    out = np.empty(m)
    out[ordem] = np.minimum(ajust, 1)
    return out


def rotulo_efeito(x):
    x = abs(x)
    return "desprezível" if x < 0.1 else "pequeno" if x < 0.3 else "médio" if x < 0.5 else "grande"


def cramers_v(tabela):
    chi2, p, _, _ = stats.chi2_contingency(tabela)
    n = tabela.to_numpy().sum()
    k = min(tabela.shape) - 1
    return chi2, p, np.sqrt(chi2 / (n * k)) if k > 0 else np.nan

In [0]:
# Visão geral: volume mensal (base inteira) e taxa de atraso (só desenvolvimento)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.2))
vol = base[na_janela].groupby("ano_mes").size()
axes[0].bar(vol.index, vol.values, color=["#9DB4D8" if m < INICIO_TESTE[:7] else "#DDDDDD" for m in vol.index])
axes[0].set_title("Pedidos por mês (cinza = período de teste, fora da EDA)")
axes[0].tick_params(axis="x", rotation=70)
taxa_mes = eda.groupby("ano_mes")["atrasou"].mean()
axes[1].plot(taxa_mes.index, taxa_mes.values, marker="o", color="#C44E52")
axes[1].yaxis.set_major_formatter(PCT)
axes[1].set_title("Taxa de atraso por mês da compra (desenvolvimento)")
axes[1].tick_params(axis="x", rotation=70)
mostrar(fig, "eda_01_temporal")

# Atraso por faixas de distância, prazo prometido e frete relativo
faixas = {
    "distancia_km": ([0, 100, 500, 1000, 2000, 1e5], ["<100", "100-500", "500-1k", "1k-2k", "2k+"]),
    "prazo_prometido_dias": ([0, 10, 20, 30, 45, 1e4], ["<10", "10-20", "20-30", "30-45", "45+"]),
    "frete_pct_ticket": ([0, 0.1, 0.2, 0.3, 0.5, 1.01], ["<10%", "10-20%", "20-30%", "30-50%", "50%+"]),
}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, (var, (cortes, rot)) in zip(axes, faixas.items()):
    t = eda.groupby(pd.cut(eda[var], cortes, labels=rot, include_lowest=True), observed=True)["atrasou"].agg(["mean", "size"])
    ax.bar(t.index.astype(str), t["mean"], color="#C44E52")
    for i, (m, n) in enumerate(zip(t["mean"], t["size"])):
        ax.annotate(f"n={n:,}", (i, m), ha="center", va="bottom", fontsize=8)
    ax.yaxis.set_major_formatter(PCT)
    ax.set_title(f"Taxa de atraso por {var}")
mostrar(fig, "eda_02_faixas")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
rota = eda.pivot_table(index="regiao_vendedor", columns="regiao_cliente", values="atrasou", aggfunc="mean")
n_rota = eda.pivot_table(index="regiao_vendedor", columns="regiao_cliente", values="atrasou", aggfunc="count")
sns.heatmap(rota.where(n_rota >= 100), annot=True, fmt=".0%", cmap="Reds", ax=axes[0])
axes[0].set_title("Atraso por rota (vazio = menos de 100 pedidos)")
cat = eda.groupby("categoria")["atrasou"].agg(["mean", "size"]).query("size >= 300").sort_values("mean")
axes[1].barh(cat.index[-15:], cat["mean"].iloc[-15:], color="#8172B2")
axes[1].xaxis.set_major_formatter(PCT)
axes[1].set_title("15 categorias com maior atraso (300+ pedidos)")
mostrar(fig, "eda_03_rota_categoria")

### Testes de hipótese: atraso

In [0]:
# Qui-quadrado + V de Cramér (categóricas). Categorias raras são agrupadas para evitar células com contagem baixa.
eda["rota_regiao"] = eda["regiao_vendedor"] + ">" + eda["regiao_cliente"]
eda["frete_gratis"] = (eda["valor_frete"] == 0).astype(int)
top_cat = eda["categoria"].value_counts().index[:25]
eda["categoria_top"] = eda["categoria"].where(eda["categoria"].isin(top_cat), "outras")
linhas = []
for var in ["customer_state", "seller_state", "rota_regiao", "categoria_top", "tipo_pagamento", "mesmo_estado", "frete_gratis"]:
    chi2, p, v = cramers_v(pd.crosstab(eda[var], eda["atrasou"]))
    linhas.append({"variável": var, "teste": "qui-quadrado", "estatística": chi2, "p": p,
                   "efeito": v, "medida": "V de Cramér"})

# Mann-Whitney U + correlação rank-biserial (numéricas): atrasados x no prazo
for var in ["valor_frete", "frete_pct_ticket", "valor_produtos", "distancia_km", "prazo_prometido_dias",
            "peso_total_g", "parcelas", "n_itens"]:
    a1 = eda.loc[eda["atrasou"] == 1, var].dropna()
    a0 = eda.loc[eda["atrasou"] == 0, var].dropna()
    u, p = stats.mannwhitneyu(a1, a0, alternative="two-sided")
    linhas.append({"variável": var, "teste": "Mann-Whitney U", "estatística": u, "p": p,
                   "efeito": 2 * u / (len(a1) * len(a0)) - 1, "medida": "rank-biserial (+ = maior nos atrasados)",
                   "mediana_atrasados": a1.median(), "mediana_no_prazo": a0.median()})

testes_atraso = pd.DataFrame(linhas)
testes_atraso["p_holm"] = holm(testes_atraso["p"])
testes_atraso["rejeita_H0 (5%)"] = testes_atraso["p_holm"] < 0.05
testes_atraso["magnitude"] = testes_atraso["efeito"].map(rotulo_efeito)
display(testes_atraso.round(4))

In [0]:
# Correlações de Pearson e Spearman com os dias de atraso (entregues). Spearman é a leitura principal:
# as variáveis são assimétricas (frete, preço, peso) e a relação não precisa ser linear.
ent = eda[eda["entregue"] == 1]
vars_corr = ["valor_frete", "frete_pct_ticket", "valor_produtos", "distancia_km", "prazo_prometido_dias", "peso_total_g"]
corr = pd.DataFrame({
    "pearson": [stats.pearsonr(ent[v].fillna(ent[v].median()), ent["atraso_dias"])[0] for v in vars_corr],
    "spearman": [stats.spearmanr(ent[v], ent["atraso_dias"], nan_policy="omit")[0] for v in vars_corr],
}, index=vars_corr)
corr["magnitude (spearman)"] = corr["spearman"].map(rotulo_efeito)
display(corr.round(3).reset_index().rename(columns={"index": "variável x atraso_dias"}))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(ent[vars_corr + ["atraso_dias"]].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Spearman: preço, frete, logística e atraso")
mostrar(fig, "eda_04_correlacoes")

In [0]:
av = eda[eda["review_score"].notna() & (eda["entregue"] == 1)].copy()
av["faixa_atraso"] = pd.cut(av["atraso_dias"], [-200, -15, -7, 0, 3, 7, 15, 500],
                            labels=["15+ antes", "7-15 antes", "0-7 antes", "1-3 atraso", "4-7 atraso",
                                    "8-15 atraso", "15+ atraso"])

fig, axes = plt.subplots(1, 3, figsize=(18, 4.3))
av["review_score"].value_counts(normalize=True).sort_index().plot.bar(ax=axes[0], rot=0, color="#4C72B0")
axes[0].yaxis.set_major_formatter(PCT)
axes[0].set_title("Distribuição das notas")
av.groupby("faixa_atraso", observed=True)["review_negativa"].mean().plot.bar(ax=axes[1], color="#C44E52", rot=30)
axes[1].yaxis.set_major_formatter(PCT)
axes[1].set_title("% de notas 1-2 por dias de atraso")
av.groupby("atrasou")["review_score"].mean().rename({0: "no prazo", 1: "atrasado"}).plot.bar(ax=axes[2], rot=0,
                                                                                             color=["#55A868", "#C44E52"])
axes[2].set_title("Nota média")
mostrar(fig, "eda_05_satisfacao")

s1, s0 = av.loc[av["atrasou"] == 1, "review_score"], av.loc[av["atrasou"] == 0, "review_score"]
u, p_u = stats.mannwhitneyu(s1, s0, alternative="two-sided")
chi2, p_chi, v = cramers_v(pd.crosstab(av["atrasou"], av["review_negativa"]))
kw = stats.kruskal(*[g["review_score"].to_numpy() for _, g in av.groupby("faixa_atraso", observed=True)])
k = av["faixa_atraso"].nunique()
rho_atraso = stats.spearmanr(av["atraso_dias"], av["review_score"])
no_prazo = av[av["atrasou"] == 0]
rho_frete = stats.spearmanr(no_prazo["frete_pct_ticket"], no_prazo["review_score"], nan_policy="omit")
rho_preco = stats.spearmanr(no_prazo["valor_produtos"], no_prazo["review_score"], nan_policy="omit")

testes_satisf = pd.DataFrame([
    ["H5", "Nota: atrasado x no prazo", "Mann-Whitney U", p_u, 2 * u / (len(s1) * len(s0)) - 1, "rank-biserial"],
    ["H5", "Atraso x nota negativa (1-2)", "Qui-quadrado", p_chi, v, "V de Cramér"],
    ["H5", "Nota entre faixas de atraso", "Kruskal-Wallis", kw.pvalue, (kw.statistic - k + 1) / (len(av) - k), "épsilon²"],
    ["H5", "Dias de atraso x nota", "Spearman", rho_atraso.pvalue, rho_atraso.statistic, "rho"],
    ["H6", "Frete/ticket x nota (só no prazo)", "Spearman", rho_frete.pvalue, rho_frete.statistic, "rho"],
    ["H6", "Valor dos produtos x nota (só no prazo)", "Spearman", rho_preco.pvalue, rho_preco.statistic, "rho"],
], columns=["hipótese", "comparação", "teste", "p", "efeito", "medida"])
testes_satisf["p_holm"] = holm(testes_satisf["p"])
testes_satisf["magnitude"] = testes_satisf["efeito"].map(rotulo_efeito)
display(testes_satisf.round(4))
print(f"Nota média: {s0.mean():.2f} no prazo x {s1.mean():.2f} atrasado | "
      f"notas 1-2: {av.loc[av['atrasou'] == 0, 'review_negativa'].mean():.0%} x {av.loc[av['atrasou'] == 1, 'review_negativa'].mean():.0%}")

## 4. Feature engineering (~50 variáveis, todas conhecidas no momento da compra)

**Regra:** uma variável só entra se o seu valor existe no instante `order_purchase_timestamp`.

- Preço, frete, prazo prometido e forma de pagamento aparecem no checkout, antes de confirmar a compra.
- Atributos do produto e localização de cliente e vendedor são cadastrais.
- **Históricos** (taxa de atraso da rota, do vendedor, da plataforma) usam só entregas concluídas **estritamente antes** de cada compra, via `merge_asof`. Nenhum pedido vê o próprio resultado nem o futuro.
- Duas razões (preço relativo à categoria e frete relativo à rota) dependem de medianas. Elas são calculadas **dentro do pipeline**, ajustadas só no treino de cada fold.

Ficam de fora, por serem posteriores à compra: tempo de aprovação, datas de postagem e entrega, `shipping_limit_date` (pode ser redefinida depois da aprovação; na dúvida, fica fora), status do pedido e qualquer dado da avaliação.

In [0]:
def _acumulados(eventos, chave):
    ev = eventos.sort_values("_t", kind="mergesort")
    g = ev.groupby(chave, sort=False)
    out = ev[[chave, "_t"]].copy()
    out["_n"] = g.cumcount() + 1
    out["_s_atraso"] = g["atrasou"].cumsum()
    out["_s_dias"] = g["dias_entrega"].cumsum()
    return out


def historico_asof(ped, chave, janela_dias=None, minimo=1):
    """Estatísticas das entregas concluídas ESTRITAMENTE antes de cada compra, por `chave`.
    Com janela_dias, só as concluídas nos últimos N dias antes da compra."""
    ev = ped.loc[ped["entregue"] == 1, [chave, "order_delivered_customer_date", "atrasou", "dias_entrega"]].copy()
    ev = ev.rename(columns={"order_delivered_customer_date": "_t"})
    ev["dias_entrega"] = ev["dias_entrega"].clip(lower=0, upper=60)
    acc = _acumulados(ev, chave)
    cols = ["_n", "_s_atraso", "_s_dias"]

    def ate(deslocamento, incluir_igual):
        q = ped[[chave, "order_purchase_timestamp"]].copy()
        q["_t"] = q["order_purchase_timestamp"] - deslocamento
        q["_idx"] = np.arange(len(q))
        q = q.sort_values("_t", kind="mergesort")
        h = pd.merge_asof(q, acc, on="_t", by=chave, direction="backward", allow_exact_matches=incluir_igual)
        return h.sort_values("_idx")[cols].fillna(0).to_numpy()

    h = ate(pd.Timedelta(0), False)                       # tudo o que terminou antes da compra
    if janela_dias is not None:                           # menos o que terminou antes de (compra - janela)
        h = h - ate(pd.Timedelta(days=janela_dias), True)
    n = h[:, 0]
    ok = n >= minimo
    with np.errstate(invalid="ignore", divide="ignore"):
        return pd.DataFrame({"n": n, "taxa_atraso": np.where(ok, h[:, 1] / n, np.nan),
                             "dias_entrega": np.where(ok, h[:, 2] / n, np.nan)}, index=ped.index)


def contagem_anterior(ped, chave, janela_dias):
    """Nº de compras com a mesma chave nos últimos N dias, estritamente antes de cada compra."""
    out = np.zeros(len(ped))
    t_all = ped["order_purchase_timestamp"].to_numpy()
    for _, pos in ped.groupby(chave, sort=False).indices.items():
        t = t_all[pos]
        ordenado = np.sort(t)
        out[pos] = (np.searchsorted(ordenado, t, "left")
                    - np.searchsorted(ordenado, t - np.timedelta64(janela_dias, "D"), "left"))
    return out


def criar_features(ped):
    ped = ped.sort_values("order_purchase_timestamp", kind="mergesort").copy()
    ts = ped["order_purchase_timestamp"]

    # ---------------------------------------------------------- Preço e desconto
    ped["dif_pago_vs_ticket_pct"] = (ped["valor_pago"] - ped["ticket_total"]) / ped["ticket_total"]
    ped["usou_voucher"] = (ped["valor_voucher"].fillna(0) > 0).astype(int)
    ped["voucher_pct"] = ped["valor_voucher"] / ped["valor_pago"].where(ped["valor_pago"] > 0)
    ped["valor_parcela"] = ped["valor_pago"] / ped["parcelas"].clip(lower=1)

    # ---------------------------------------------------------- Frete e prazos
    ped["peso_cubado_kg"] = np.fmax(ped["peso_total_g"] / 1000, ped["volume_total_cm3"] / 6000)   # fator Correios
    ped["frete_por_kg"] = ped["valor_frete"] / ped["peso_cubado_kg"].clip(lower=0.1)
    ped["frete_gratis"] = (ped["valor_frete"] == 0).astype(int)

    # ---------------------------------------------------------- Logística e localidade
    ped["mesma_cidade"] = ((ped["cidade_cliente"] == ped["cidade_vendedor"]).fillna(False) & (ped["mesmo_estado"] == 1)).astype(int)
    capital = ped["customer_state"].map({uf: c[0] for uf, c in CAPITAIS.items()})
    ped["cliente_em_capital"] = (ped["cidade_cliente"] == capital).fillna(False).astype(int)

    ped["_rota_uf"] = ped["seller_state"] + ">" + ped["customer_state"]
    ped["_rota_reg"] = ped["regiao_vendedor"] + ">" + ped["regiao_cliente"]
    ped["_geral"] = "BR"
    ped["_vendedor"] = ped["vendedor_principal"].fillna("NA")
    ped["_cliente"] = ped["customer_unique_id"].fillna(ped["customer_id"])

    uf = historico_asof(ped, "_rota_uf", minimo=MIN_EVENTOS_ROTA)
    reg = historico_asof(ped, "_rota_reg", minimo=MIN_EVENTOS_ROTA)
    ger = historico_asof(ped, "_geral", minimo=30)
    ped["taxa_atraso_rota_hist"] = uf["taxa_atraso"].fillna(reg["taxa_atraso"]).fillna(ger["taxa_atraso"])
    ped["entrega_rota_hist_dias"] = uf["dias_entrega"].fillna(reg["dias_entrega"]).fillna(ger["dias_entrega"])
    uf_r = historico_asof(ped, "_rota_uf", JANELA_RECENTE_DIAS, MIN_EVENTOS_RECENTE)
    reg_r = historico_asof(ped, "_rota_reg", JANELA_RECENTE_DIAS, MIN_EVENTOS_RECENTE)
    ger_r = historico_asof(ped, "_geral", JANELA_RECENTE_DIAS, MIN_EVENTOS_RECENTE)
    ped["taxa_atraso_rota_recente"] = uf_r["taxa_atraso"].fillna(reg_r["taxa_atraso"]).fillna(ger_r["taxa_atraso"])
    ped["taxa_atraso_plataforma_recente"] = ger_r["taxa_atraso"]
    destino = historico_asof(ped, "customer_state", minimo=MIN_EVENTOS_ROTA)
    ped["taxa_atraso_destino_hist"] = destino["taxa_atraso"].fillna(ger["taxa_atraso"])
    ped["pedidos_plataforma_7d"] = contagem_anterior(ped, "_geral", 7)

    # Folgas: prazo prometido menos o tempo que a rota/vendedor costumava levar
    ped["folga_prometida"] = ped["prazo_prometido_dias"] - ped["entrega_rota_hist_dias"]

    # ---------------------------------------------------------- Produto e vendedor
    vend = historico_asof(ped, "_vendedor", minimo=MIN_EVENTOS_VENDEDOR)
    vend_r = historico_asof(ped, "_vendedor", JANELA_RECENTE_DIAS, MIN_EVENTOS_VENDEDOR)
    ped["vendedor_entregas_concluidas"] = vend["n"]
    ped["vendedor_taxa_atraso_hist"] = vend["taxa_atraso"]
    ped["vendedor_dias_entrega_hist"] = vend["dias_entrega"]
    ped["vendedor_taxa_atraso_recente"] = vend_r["taxa_atraso"]
    ped["vendedor_pedidos_anteriores"] = ped.groupby("_vendedor")["order_purchase_timestamp"].rank(method="min") - 1
    ped["vendedor_pedidos_30d"] = contagem_anterior(ped, "_vendedor", 30)
    primeira_venda = ped.groupby("_vendedor")["order_purchase_timestamp"].transform("min")
    ped["vendedor_idade_dias"] = dias(ts - primeira_venda)
    ped["folga_vs_vendedor"] = ped["prazo_prometido_dias"] - ped["vendedor_dias_entrega_hist"]

    # ---------------------------------------------------------- Calendário e cliente
    ped["mes"] = ts.dt.month
    ped["dia_semana"] = ts.dt.dayofweek
    ped["hora"] = ts.dt.hour
    ped["fim_de_semana"] = (ped["dia_semana"] >= 5).astype(int)
    ped["cliente_recorrente"] = (ped.groupby("_cliente")["order_purchase_timestamp"].rank(method="min") > 1).astype(int)

    return ped.drop(columns=["_rota_uf", "_rota_reg", "_geral", "_vendedor", "_cliente"])


t0 = time.time()
base_feat = criar_features(base)
print(f"Features criadas em {time.time() - t0:.0f}s")

### Dicionário de dados
Cada variável tem grupo, momento em que é conhecida e descrição. O modelo lê a lista de features **deste dicionário**, filtrando `momento == "compra"`. Assim, uma variável pós-compra não consegue entrar no modelo por engano.

In [0]:
D = []   # (feature, grupo, momento, tipo, descrição)
def _d(grupo, momento, tipo, itens):
    D.extend((f, grupo, momento, tipo, desc) for f, desc in itens)

_d("Preço e desconto", "compra", "numérica", [
    ("valor_produtos", "Soma dos preços dos itens"),
    ("preco_medio_item", "Preço médio por item"),
    ("n_itens", "Quantidade de itens no pedido"),
    ("ticket_total", "Produtos + frete"),
    ("dif_pago_vs_ticket_pct", "(valor pago - ticket) / ticket: negativo = desconto/voucher, positivo = juros do parcelamento"),
    ("usou_voucher", "Pagou parte com voucher (escolhido no checkout)"),
    ("voucher_pct", "Parcela do pagamento feita com voucher"),
    ("parcelas", "Número de parcelas escolhido"),
    ("valor_parcela", "Valor pago / parcelas"),
    ("preco_rel_categoria", "Valor dos produtos / mediana da categoria (calculada no pipeline, só com o treino)"),
])
_d("Preço e desconto", "compra", "categórica", [("tipo_pagamento", "Forma de pagamento de maior valor")])
_d("Frete e prazos estimados", "compra", "numérica", [
    ("valor_frete", "Frete cotado (soma dos itens)"),
    ("frete_pct_ticket", "Frete / ticket total"),
    ("frete_por_kg", "Frete / peso cubado"),
    ("frete_gratis", "Frete zerado"),
    ("frete_rel_rota", "Frete / mediana do frete da rota UF>UF (calculada no pipeline, só com o treino)"),
    ("prazo_prometido_dias", "Dias entre a compra e a data prometida ao cliente"),
    ("entrega_rota_hist_dias", "Tempo médio de entrega da rota nas entregas concluídas antes da compra"),
    ("folga_prometida", "Prazo prometido - tempo histórico da rota"),
    ("folga_vs_vendedor", "Prazo prometido - tempo histórico de entrega do vendedor"),
])
_d("Logística e localidade", "compra", "categórica", [
    ("customer_state", "UF do cliente"), ("seller_state", "UF do vendedor"),
    ("regiao_cliente", "Região do cliente"), ("regiao_vendedor", "Região do vendedor"),
])
_d("Logística e localidade", "compra", "numérica", [
    ("mesmo_estado", "Vendedor e cliente na mesma UF"),
    ("mesma_regiao", "Vendedor e cliente na mesma região"),
    ("mesma_cidade", "Vendedor e cliente na mesma cidade"),
    ("cliente_em_capital", "Cliente mora na capital da UF"),
    ("distancia_km", "Distância vendedor-cliente (CEP; capital da UF se faltar geolocalização)"),
    ("taxa_atraso_rota_hist", "Taxa de atraso da rota UF>UF antes da compra (região>região ou geral se poucos casos)"),
    ("taxa_atraso_rota_recente", f"Idem, só entregas concluídas nos últimos {JANELA_RECENTE_DIAS} dias"),
    ("taxa_atraso_destino_hist", "Taxa de atraso para a UF do cliente antes da compra"),
    ("taxa_atraso_plataforma_recente", f"Taxa de atraso de toda a plataforma nos últimos {JANELA_RECENTE_DIAS} dias (greves, picos)"),
    ("pedidos_plataforma_7d", "Volume de compras na plataforma nos 7 dias anteriores (pressão logística)"),
])
_d("Produto e vendedor", "compra", "categórica", [("categoria", "Categoria do item principal")])
_d("Produto e vendedor", "compra", "numérica", [
    ("peso_total_g", "Peso total dos itens"),
    ("volume_total_cm3", "Volume total dos itens"),
    ("peso_cubado_kg", "Maior entre peso real e volume/6000"),
    ("maior_dimensao_cm", "Maior dimensão entre os itens"),
    ("fotos_media", "Fotos por anúncio (média)"),
    ("tam_descricao_media", "Tamanho da descrição do anúncio (média)"),
    ("n_vendedores", "Vendedores distintos no pedido"),
    ("vendedor_pedidos_anteriores", "Pedidos do vendedor antes desta compra"),
    ("vendedor_pedidos_30d", "Pedidos do vendedor nos 30 dias anteriores"),
    ("vendedor_idade_dias", "Dias desde a primeira venda do vendedor"),
    ("vendedor_entregas_concluidas", "Entregas do vendedor concluídas antes da compra"),
    ("vendedor_taxa_atraso_hist", "Taxa de atraso do vendedor antes da compra (nulo se < 3 entregas)"),
    ("vendedor_taxa_atraso_recente", f"Idem, últimos {JANELA_RECENTE_DIAS} dias"),
    ("vendedor_dias_entrega_hist", "Tempo médio de entrega do vendedor antes da compra"),
])
_d("Calendário e cliente", "compra", "numérica", [
    ("mes", "Mês da compra"), ("dia_semana", "Dia da semana (0 = segunda)"), ("hora", "Hora da compra"),
    ("fim_de_semana", "Compra no sábado ou domingo"),
    ("cliente_recorrente", "Cliente já comprou antes"),
])
# Pós-compra: documentadas para deixar explícito por que NÃO entram
_d("Pós-compra (proibida no modelo)", "pós-compra", "alvo", [
    ("atrasou", "ALVO: entrega após a data prometida"),
    ("atraso_dias", "Dias de atraso (define o alvo)"),
])
_d("Pós-compra (proibida no modelo)", "pós-compra", "numérica", [
    ("horas_aprovacao", "Horas entre compra e aprovação do pagamento"),
    ("dias_aprovacao_postagem", "Dias entre aprovação e postagem"),
    ("postou_apos_limite", "Vendedor postou depois do limite"),
    ("dias_transporte", "Dias entre postagem e entrega"),
    ("dias_entrega", "Dias entre compra e entrega"),
    ("entregue", "Pedido entregue"),
    ("review_score", "Nota da avaliação"),
    ("review_negativa", "Nota 1 ou 2"),
    ("tem_comentario", "Avaliação com comentário"),
    ("data_limite_postagem", "shipping_limit_date: pode ser redefinida após a aprovação"),
])
dicionario = pd.DataFrame(D, columns=["feature", "grupo", "momento", "tipo", "descricao"])
dicionario["calculada_no_pipeline"] = dicionario["feature"].isin(["preco_rel_categoria", "frete_rel_rota"])

FEATURES_COMPRA = dicionario.query("momento == 'compra'")
CATEGORICAS = FEATURES_COMPRA.query("tipo == 'categórica'")["feature"].tolist()
NUMERICAS = FEATURES_COMPRA.query("tipo == 'numérica'")["feature"].tolist()
NUMERICAS_X = [f for f in NUMERICAS if f not in ("preco_rel_categoria", "frete_rel_rota")]   # colunas de entrada
FEATURES_X = CATEGORICAS + NUMERICAS_X
GRUPO = dict(zip(dicionario["feature"], dicionario["grupo"]))

# Travas: nenhuma feature do modelo pode ser pós-compra, e todas precisam existir na base
proibidas = set(dicionario.query("momento != 'compra'")["feature"])
assert not proibidas & set(FEATURES_X), f"vazamento: {proibidas & set(FEATURES_X)}"
faltando = [f for f in FEATURES_X if f not in base_feat.columns]
assert not faltando, f"features não construídas: {faltando}"

print(f"{len(FEATURES_COMPRA)} features do momento da compra ({len(CATEGORICAS)} categóricas, {len(NUMERICAS)} numéricas)")
display(FEATURES_COMPRA.groupby("grupo").size().rename("nº de features").reset_index())
display(dicionario)

### Auditoria automática contra vazamento
1. **Recalcular na força bruta** o histórico do vendedor para 300 pedidos sorteados, filtrando explicitamente "entregue antes da compra", e conferir se bate com o `merge_asof`.
2. **Teste de uma variável só:** nenhuma feature sozinha deveria prever atraso quase perfeitamente. Se alguma passar de ROC-AUC 0,85 ou PR-AUC 0,5, é sinal de vazamento. `atraso_dias` (que define o alvo) entra como **controle positivo**: o teste precisa pegá-la.

Esse teste só detecta vazamento grosseiro. Um vazamento sutil (ex.: uma média calculada com dados futuros) melhora o modelo sem chegar perto da perfeição; ele é evitado pela construção 

In [0]:
# 1) força bruta
rng = np.random.default_rng(SEED)
amostra = base_feat[base_feat["elegivel"] == 1].sample(300, random_state=SEED)
entregues = base_feat[base_feat["entregue"] == 1]
erros = 0
for _, r in amostra.iterrows():
    ant = entregues[(entregues["vendedor_principal"] == r["vendedor_principal"])
                    & (entregues["order_delivered_customer_date"] < r["order_purchase_timestamp"])]
    erros += int(len(ant) != r["vendedor_entregas_concluidas"])
    if len(ant) >= MIN_EVENTOS_VENDEDOR:
        erros += int(not np.isclose(ant["atrasou"].mean(), r["vendedor_taxa_atraso_hist"]))
assert erros == 0, f"{erros} divergências no histórico do vendedor"
print("Força bruta OK: o histórico do vendedor usa só entregas concluídas antes de cada compra")

# 2) poder preditivo de cada variável sozinha (período de desenvolvimento)
dev = base_feat[(base_feat["elegivel"] == 1) & (base_feat["order_purchase_timestamp"] >= DATA_INICIO) & (base_feat["order_purchase_timestamp"] < INICIO_TESTE)]
y_dev = dev["atrasou"].to_numpy()
aud = []
for f in NUMERICAS_X + ["atraso_dias"]:
    x = dev[f].fillna(dev[f].median()).to_numpy()
    roc = roc_auc_score(y_dev, x)
    sinal = 1 if roc >= 0.5 else -1
    aud.append({"feature": f, "roc_auc": max(roc, 1 - roc), "pr_auc": average_precision_score(y_dev, sinal * x)})
aud = pd.DataFrame(aud).sort_values("roc_auc", ascending=False)
aud["suspeita"] = (aud["roc_auc"] > 0.85) | (aud["pr_auc"] > 0.5)
display(aud.round(3).head(12))
assert aud.set_index("feature").loc["atraso_dias", "suspeita"], "o controle positivo deveria ser detectado"
suspeitas = aud.query("suspeita and feature != 'atraso_dias'")["feature"].tolist()
assert not suspeitas, f"features suspeitas de vazamento: {suspeitas}"
print(f"Auditoria OK (taxa base {y_dev.mean():.3f}): só o controle pós-compra foi sinalizado")

## 5. Estratégia de validação temporal

```
jan/17 ─────────── jun/17 │ jul-set/17 │ out-dez/17 │ jan-mar/18 ║ abr-ago/18
 fold 1: treino ──────────┤ validação  │            │            ║
 fold 2: treino ───────────────────────┤ validação  │            ║
 fold 3: treino ────────────────────────────────────┤ validação  ║
 final:  treino ────────────────────────────────────┤ calibração ║   TESTE (uma vez)
```

- **Folds cronológicos com janela expansível:** o modelo nunca treina com pedidos posteriores aos que valida.
- **Embargo:** um pedido de dezembro só tem resultado conhecido em janeiro. Em produção, no dia 1º de janeiro, esse rótulo ainda não existiria. Por isso o treino de cada fold exclui compras dos últimos `EMBARGO_DIAS` dias antes do corte e qualquer pedido cujo resultado só fica conhecido depois do corte.
- **Balanceamento dentro do fold:** a estratégia (nenhuma, pesos por classe ou subamostragem da classe majoritária) é um hiperparâmetro. Ela é aplicada pelo wrapper `ModeloBalanceado` dentro do `fit`, que só recebe o treino do fold. Validação, calibração e teste mantêm a proporção real de atrasos.
- **Calibração:** pesos e subamostragem distorcem as probabilidades. Por isso o modelo final recebe um calibrador sigmoide ajustado em jan-mar/2018, dados que ele não viu no treino. Atenção: se a taxa de atraso de jan-mar/2018 for muito diferente da de abr-ago/2018, o calibrador desloca as probabilidades. A tabela da seção 7 mostra o Brier com e sem calibração. Se a calibração piorar, troque `CALIBRAR` e **justifique pelo período de calibração**, nunca pelo teste.
- SMOTE não é usado: com variáveis categóricas e de histórico, ele cria pedidos que não existem (ex.: taxa de atraso de vendedor "entre" dois vendedores).

In [0]:
bm = base_feat[(base_feat["elegivel"] == 1)
               & base_feat["order_purchase_timestamp"].between(DATA_INICIO, f"{DATA_FIM} 23:59:59")].copy()
bm["atrasou"] = bm["atrasou"].astype(int)
TS = bm["order_purchase_timestamp"]


def treino_ate(corte):
    """Treino com embargo: compra antes de (corte - EMBARGO_DIAS) e resultado já conhecido no corte."""
    corte = pd.Timestamp(corte)
    m = (TS < corte - pd.Timedelta(days=EMBARGO_DIAS)) & (bm["data_conhecimento_alvo"] < corte)
    return bm[m]


def Xy(df):
    return df[FEATURES_X], df["atrasou"].to_numpy()


folds = []
for ini, fim in FOLDS_VALIDACAO:
    tr = treino_ate(ini)
    va = bm[(TS >= ini) & (TS < fim)]
    folds.append({"nome": f"valida {ini[:7]} a {pd.Timestamp(fim) - pd.Timedelta(days=1):%Y-%m}",
                  "tr": tr, "va": va})

treino_final = treino_ate(FIM_TREINO_FINAL)
calib = bm[(TS >= FIM_TREINO_FINAL) & (TS < INICIO_TESTE) & (bm["data_conhecimento_alvo"] < INICIO_TESTE)]
teste = bm[TS >= INICIO_TESTE]

resumo = [{"conjunto": f["nome"], "treino": len(f["tr"]), "atraso_treino": f["tr"]["atrasou"].mean(),
           "avaliação": len(f["va"]), "atraso_avaliação": f["va"]["atrasou"].mean()} for f in folds]
resumo += [{"conjunto": "final (calibração jan-mar/18)", "treino": len(treino_final),
            "atraso_treino": treino_final["atrasou"].mean(), "avaliação": len(calib),
            "atraso_avaliação": calib["atrasou"].mean()},
           {"conjunto": "TESTE abr-ago/18", "treino": np.nan, "atraso_treino": np.nan,
            "avaliação": len(teste), "atraso_avaliação": teste["atrasou"].mean()}]
display(pd.DataFrame(resumo).round(4))

# Garantias da divisão
assert treino_final["order_purchase_timestamp"].max() < pd.Timestamp(FIM_TREINO_FINAL)
assert calib["order_purchase_timestamp"].min() >= pd.Timestamp(FIM_TREINO_FINAL)
assert teste["order_purchase_timestamp"].min() >= pd.Timestamp(INICIO_TESTE)
assert not set(teste["order_id"]) & (set(treino_final["order_id"]) | set(calib["order_id"]))
fora = bm[(TS < pd.Timestamp(FIM_TREINO_FINAL)) & ~bm["order_id"].isin(treino_final["order_id"])]
print(f"Embargo retirou {len(fora):,} pedidos do treino final (taxa de atraso deles: {fora['atrasou'].mean():.2%})")

 Modelagem: pré-processamento, balanceamento e os 4 modelos

In [0]:
def log_com_sinal(x):
    return np.sign(x) * np.log1p(np.abs(x))


class RazoesPelaMediana(BaseEstimator, TransformerMixin):
    """Cria valor / mediana do grupo. As medianas vêm só do treino (fit); grupos com menos de
    `minimo` pedidos usam a mediana geral do treino."""
    REGRAS = (("preco_rel_categoria", "valor_produtos", ("categoria",)),
              ("frete_rel_rota", "valor_frete", ("seller_state", "customer_state")))

    def __init__(self, minimo=30):
        self.minimo = minimo

    @staticmethod
    def _chave(X, grupo):
        chave = X[grupo[0]].astype(str)
        for c in grupo[1:]:
            chave = chave + ">" + X[c].astype(str)
        return chave

    def fit(self, X, y=None):
        self.medianas_ = {}
        for nome, valor, grupo in self.REGRAS:
            chave = self._chave(X, grupo)
            g = X[valor].groupby(chave)
            med = g.median()[g.count() >= self.minimo]
            self.medianas_[nome] = (med, float(X[valor].median()))
        return self

    def transform(self, X):
        X = X.copy()
        for nome, valor, grupo in self.REGRAS:
            med, geral = self.medianas_[nome]
            chave = self._chave(X, grupo)
            ref = chave.map(med).fillna(geral).replace(0, np.nan)
            X[nome] = X[valor] / ref
        return X


class ParaCategoria(BaseEstimator, TransformerMixin):
    """Texto -> 'category' com as categorias vistas no treino (valor novo vira nulo)."""
    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        self.categorias_ = {c: sorted(X[c].dropna().unique()) for c in self.colunas}
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.colunas:
            X[c] = pd.Categorical(X[c], categories=self.categorias_[c])
        return X


class ModeloBalanceado(BaseEstimator, ClassifierMixin):
    """Aplica o balanceamento DENTRO do fit, ou seja, só nos dados de treino que o modelo recebe.
    estrategia: 'nenhum' | 'pesos' (classes com peso igual) | 'subamostragem' (negativos = razao x positivos)."""
    def __init__(self, estimador, estrategia="nenhum", razao=3.0, random_state=SEED):
        self.estimador = estimador
        self.estrategia = estrategia
        self.razao = razao
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y)
        est = clone(self.estimador)
        if self.estrategia == "subamostragem":
            rng = np.random.default_rng(self.random_state)
            pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
            n_neg = min(len(neg), int(self.razao * len(pos)))
            idx = np.sort(np.concatenate([pos, rng.choice(neg, n_neg, replace=False)]))
            est.fit(X.iloc[idx], y[idx])
        elif self.estrategia == "pesos":
            w = np.where(y == 1, len(y) / (2 * y.sum()), len(y) / (2 * (len(y) - y.sum())))
            est.fit(X, y, modelo__sample_weight=w)
        else:
            est.fit(X, y)
        self.estimador_ = est
        self.classes_ = est.classes_
        return self

    def predict_proba(self, X):
        return self.estimador_.predict_proba(X)

    def decision_function(self, X):
        """Logit da probabilidade. O calibrador sigmoide usa este score; sem ele, algumas versões do
        scikit-learn pegam colunas diferentes de predict_proba no ajuste e na previsão e invertem o ranking."""
        p = np.clip(self.predict_proba(X)[:, 1], 1e-7, 1 - 1e-7)
        return np.log(p / (1 - p))

    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] >= 0.5).astype(int)]


def prep_linear():
    num = Pipeline([("imputa", SimpleImputer(strategy="median", add_indicator=True)),
                    ("log", FunctionTransformer(log_com_sinal, feature_names_out="one-to-one")),
                    ("padroniza", StandardScaler())])
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), CATEGORICAS),
                              ("num", num, NUMERICAS)])


def prep_arvore():
    return ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1), CATEGORICAS),
        ("num", SimpleImputer(strategy="median", add_indicator=True), NUMERICAS)])


def construir(nome):
    if nome == "Regressão Logística":
        etapas = [("prep", prep_linear()), ("modelo", LogisticRegression(max_iter=3000))]
    elif nome == "Random Forest":
        etapas = [("prep", prep_arvore()),
                  ("modelo", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED))]
    elif nome == "Gradient Boosting":
        etapas = [("prep", prep_arvore()), ("modelo", GradientBoostingClassifier(random_state=SEED))]
    elif nome == "LightGBM":
        etapas = [("cat", ParaCategoria(CATEGORICAS)),
                  ("modelo", LGBMClassifier(random_state=SEED, verbose=-1, n_jobs=-1, subsample_freq=1))]
    else:
        raise ValueError(nome)
    return ModeloBalanceado(Pipeline([("razoes", RazoesPelaMediana())] + etapas))


BALANCEAMENTO = ["nenhum", "pesos", "subamostragem"]
ESPACOS = {
    "Regressão Logística": {"estimador__modelo__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1.0],
                            "estrategia": BALANCEAMENTO},
    "Random Forest": {"estimador__modelo__max_depth": [8, 12, 16, None],
                      "estimador__modelo__min_samples_leaf": [20, 50, 100, 200],
                      "estimador__modelo__max_features": ["sqrt", 0.3, 0.5],
                      "estrategia": BALANCEAMENTO},
    "Gradient Boosting": {"estimador__modelo__n_estimators": [100, 200, 300],
                          "estimador__modelo__learning_rate": [0.03, 0.05, 0.1],
                          "estimador__modelo__max_depth": [2, 3, 4],
                          "estimador__modelo__min_samples_leaf": [50, 100, 200],
                          "estimador__modelo__subsample": [0.7, 0.85],
                          "estimador__modelo__max_features": [0.5, "sqrt"],
                          "estrategia": BALANCEAMENTO},
    "LightGBM": {"estimador__modelo__learning_rate": [0.02, 0.05, 0.1],
                 "estimador__modelo__n_estimators": [200, 400, 600],
                 "estimador__modelo__num_leaves": [15, 31, 63],
                 "estimador__modelo__min_child_samples": [50, 100, 200],
                 "estimador__modelo__reg_lambda": [0.0, 1.0, 5.0],
                 "estimador__modelo__colsample_bytree": [0.6, 0.8, 1.0],
                 "estimador__modelo__subsample": [0.7, 1.0],
                 "estrategia": BALANCEAMENTO},
}
INTERPRETABILIDADE = {"Regressão Logística": 1.0, "Random Forest": 0.5, "Gradient Boosting": 0.5, "LightGBM": 0.5}
MODELOS = list(ESPACOS)

### Treino final, calibração e avaliação única no teste

In [0]:
import pickle
ARQ_BUSCA = os.path.join(SAIDA_DIR, "buscas.pkl")
if os.path.exists(ARQ_BUSCA) and not MODO_RAPIDO:
    with open(ARQ_BUSCA, "rb") as f:
        buscas = pickle.load(f)
    print("Busca carregada do arquivo salvo; apague buscas.pkl para refazer.")
else:
    dados_folds = [(*Xy(f["tr"]), *Xy(f["va"])) for f in folds]
    buscas = {}
    for nome in MODELOS:
        combinacoes = list(ParameterSampler(ESPACOS[nome], n_iter=N_ITER_BUSCA, random_state=SEED))
        linhas_busca = []
        for i, params in enumerate(combinacoes, 1):
            scores = []
            for Xtr_, ytr_, Xva_, yva_ in dados_folds:
                m = construir(nome).set_params(**params).fit(Xtr_, ytr_)
                scores.append(average_precision_score(yva_, m.predict_proba(Xva_)[:, 1]))
            linhas_busca.append({"params": params, "pr_auc_cv": float(np.mean(scores))})
            print(f"  {nome} [{i}/{len(combinacoes)}] PR-AUC CV {np.mean(scores):.4f}")
        buscas[nome] = (pd.DataFrame(linhas_busca)
                        .sort_values("pr_auc_cv", ascending=False).reset_index(drop=True))
        print(f"{nome}: melhor PR-AUC CV {buscas[nome]['pr_auc_cv'].iloc[0]:.4f} entre {len(linhas_busca)} combinações")
    with open(ARQ_BUSCA, "wb") as f:
        pickle.dump(buscas, f)
    print(f"Busca concluída e salva em {ARQ_BUSCA}")
import cloudpickle


def calibrar(modelo, X, y):
    """Ajusta só o calibrador; o modelo fica congelado."""
    try:
        from sklearn.frozen import FrozenEstimator   # scikit-learn >= 1.6

        class Congelado(FrozenEstimator):
            """sklearn 1.7: ModeloBalanceado herda BaseEstimator antes de ClassifierMixin, então
            is_classifier() devolve False e o CalibratedClassifierCV recusa o modelo ("Got a regressor").
            Restaurar a tag de classificador congela o modelo sem perder o tipo."""
            def __sklearn_tags__(self):
                tags = super().__sklearn_tags__()
                tags.estimator_type = "classifier"
                return tags

        return CalibratedClassifierCV(Congelado(modelo), method="sigmoid").fit(X, y)
    except ImportError:
        return CalibratedClassifierCV(modelo, method="sigmoid", cv="prefit").fit(X, y)


def modelo_base(modelo):
    """Pipeline original, sem calibrador e sem wrapper de balanceamento (para coeficientes e SHAP)."""
    if isinstance(modelo, CalibratedClassifierCV):
        est = modelo.calibrated_classifiers_[0].estimator
        modelo = getattr(est, "estimator", est)
    return modelo.estimador_ if isinstance(modelo, ModeloBalanceado) else modelo


def limiar_f1(y, p):
    """Limiar que maximiza o F1 (escolhido na calibração, aplicado depois no teste)."""
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-12, None)
    return float(th[np.nanargmax(f1)])


def avaliar(y, p, datas, n_boot=500):
    rng = np.random.default_rng(SEED)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() > 0:
            boots.append(average_precision_score(y[i], p[i]))
    meses = pd.Series(datas).dt.to_period("M").to_numpy()
    pr_mes = []
    for mes in np.unique(meses):
        sel = meses == mes
        if y[sel].sum() > 0:
            pr_mes.append(average_precision_score(y[sel], p[sel]))
    top = p >= np.quantile(p, 0.9)
    return {
        "pr_auc": average_precision_score(y, p),
        "pr_auc_ic95": (np.percentile(boots, 2.5), np.percentile(boots, 97.5)),
        "roc_auc": roc_auc_score(y, p),
        "brier": brier_score_loss(y, p),
        "taxa_base": y.mean(),
        "precisao_top10": y[top].mean(),
        "lift_top10": y[top].mean() / y.mean(),
        "pr_auc_por_mes": pr_mes,
        "variacao_mensal": np.std(pr_mes) / np.mean(pr_mes),
    }


Xtr, ytr = Xy(treino_final)
Xcal, ycal = Xy(calib)
Xte, yte = Xy(teste)
datas_te = teste["order_purchase_timestamp"].to_numpy()

finais, res, prev = {}, {}, {}
for nome in MODELOS:
    params = buscas[nome]["params"].iloc[0]
    t0 = time.time()
    base_m = construir(nome).set_params(**params).fit(Xtr, ytr)
    t_treino = time.time() - t0
    modelo = calibrar(base_m, Xcal, ycal) if CALIBRAR else base_m
    t0 = time.time()
    p = modelo.predict_proba(Xte)[:, 1]
    ms_1000 = (time.time() - t0) / len(yte) * 1e6
    p_sem = base_m.predict_proba(Xte)[:, 1]
    assert stats.spearmanr(p, p_sem).statistic > 0.99, f"{nome}: a calibração alterou a ordem dos scores"
    p_cal = modelo.predict_proba(Xcal)[:, 1]
    m = avaliar(yte, p, datas_te)
    m.update({
        "params": params,
        "pr_auc_cv": buscas[nome]["pr_auc_cv"].iloc[0],
        "pr_auc_treino": average_precision_score(ytr, base_m.predict_proba(Xtr)[:, 1]),
        "brier_sem_calibracao": brier_score_loss(yte, p_sem),
        "prob_media": p.mean(),
        "prob_media_sem_calibracao": p_sem.mean(),
        "limiar": limiar_f1(ycal, p_cal),
        "segundos_treino": t_treino,
        "ms_por_1000_previsoes": ms_1000,
        "tamanho_mb": len(cloudpickle.dumps(modelo)) / 1e6,
        "interpretabilidade": INTERPRETABILIDADE[nome],
    })
    finais[nome], res[nome], prev[nome] = modelo, m, p
    print(f"{nome:20s} TESTE PR-AUC {m['pr_auc']:.4f} "
          f"(IC95 {m['pr_auc_ic95'][0]:.4f}-{m['pr_auc_ic95'][1]:.4f}) | "
          f"ROC-AUC {m['roc_auc']:.4f} | Brier {m['brier_sem_calibracao']:.4f} -> {m['brier']:.4f} | "
          f"top10% {m['precisao_top10']:.1%}")

# Referência sem machine learning: taxa histórica de atraso da rota
p_ref = teste["taxa_atraso_rota_hist"].fillna(ytr.mean()).to_numpy()
ref = avaliar(yte, p_ref, datas_te)
print(f"\nReferência (taxa da rota) PR-AUC {ref['pr_auc']:.4f} | acaso = taxa base {ref['taxa_base']:.4f}")

Checkpoint 

In [0]:
# Treino final com checkpoint: cada modelo concluído é salvo em disco. Se a sessão cair ou a célula
# for interrompida, ao rodar de novo os modelos prontos são carregados e só os que faltam são treinados.
import pickle
import cloudpickle


def calibrar(modelo, X, y):
    """Ajusta só o calibrador; o modelo fica congelado."""
    try:
        from sklearn.frozen import FrozenEstimator   # scikit-learn >= 1.6

        class Congelado(FrozenEstimator):
            """Mantém a tag de classificador (sklearn 1.7 a trata o wrapper como regressor)."""
            def __sklearn_tags__(self):
                tags = super().__sklearn_tags__()
                tags.estimator_type = "classifier"
                return tags

        return CalibratedClassifierCV(Congelado(modelo), method="sigmoid").fit(X, y)
    except ImportError:
        return CalibratedClassifierCV(modelo, method="sigmoid", cv="prefit").fit(X, y)


def modelo_base(modelo):
    """Pipeline original, sem calibrador e sem wrapper de balanceamento."""
    if isinstance(modelo, CalibratedClassifierCV):
        est = modelo.calibrated_classifiers_[0].estimator
        modelo = getattr(est, "estimator", est)
    return modelo.estimador_ if isinstance(modelo, ModeloBalanceado) else modelo


def limiar_f1(y, p):
    """Limiar que maximiza o F1, escolhido na calibração (nunca no teste)."""
    pr, rc, th = precision_recall_curve(y, p)
    f1 = 2 * pr[:-1] * rc[:-1] / np.clip(pr[:-1] + rc[:-1], 1e-12, None)
    return float(th[np.nanargmax(f1)])


def avaliar(y, p, datas, n_boot=200):
    rng = np.random.default_rng(SEED)
    boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() > 0:
            boots.append(average_precision_score(y[i], p[i]))
    meses = pd.Series(datas).dt.to_period("M").to_numpy()
    pr_mes = []
    for mes in np.unique(meses):
        sel = meses == mes
        if y[sel].sum() > 0:
            pr_mes.append(average_precision_score(y[sel], p[sel]))
    top = p >= np.quantile(p, 0.9)
    return {
        "pr_auc": average_precision_score(y, p),
        "pr_auc_ic95": (np.percentile(boots, 2.5), np.percentile(boots, 97.5)),
        "roc_auc": roc_auc_score(y, p),
        "brier": brier_score_loss(y, p),
        "taxa_base": y.mean(),
        "precisao_top10": y[top].mean(),
        "lift_top10": y[top].mean() / y.mean(),
        "pr_auc_por_mes": pr_mes,
        "variacao_mensal": np.std(pr_mes) / np.mean(pr_mes),
    }


def feature_original(nome_transformado):
    """'cat__categoria_moveis' -> 'categoria'; 'num__missingindicator_voucher_pct' -> 'voucher_pct'."""
    n = nome_transformado.split("__", 1)[-1].replace("missingindicator_", "")
    candidatos = [f for f in CATEGORICAS + NUMERICAS if n == f or n.startswith(f + "_")]
    return max(candidatos, key=len) if candidatos else n


def importancia_nativa(modelo):
    """Importância do próprio modelo, sem reprocessar dados (custo zero):
    |coeficiente| na Regressão Logística, ganho de impureza nas árvores, ganho no LightGBM."""
    pipe = modelo_base(modelo)
    est = pipe.named_steps["modelo"]
    if "prep" in pipe.named_steps:
        nomes = pipe.named_steps["prep"].get_feature_names_out()
    else:
        nomes = getattr(est, "feature_name_", None) or list(getattr(est, "feature_names_in_"))
    if hasattr(est, "coef_"):
        valores = np.abs(est.coef_[0])
    elif hasattr(est, "booster_"):
        valores = est.booster_.feature_importance(importance_type="gain")
    else:
        valores = est.feature_importances_
    imp = pd.Series(valores, index=[feature_original(n) for n in nomes]).groupby(level=0).sum()
    return imp / imp.sum()


Xtr, ytr = Xy(treino_final)
Xcal, ycal = Xy(calib)
Xte, yte = Xy(teste)
datas_te = teste["order_purchase_timestamp"].to_numpy()
DIR_CHECKPOINT = os.path.join(SAIDA_DIR, "checkpoint_finais")
os.makedirs(DIR_CHECKPOINT, exist_ok=True)

finais, res, prev, imp_nativa = {}, {}, {}, {}
for nome in MODELOS:
    arq = os.path.join(DIR_CHECKPOINT, f"{nome.replace(' ', '_')}.pkl")
    if os.path.exists(arq):
        with open(arq, "rb") as f:
            res[nome], prev[nome], imp_nativa[nome] = pickle.load(f)
        print(f"{nome:20s} carregado do checkpoint")
        continue
    params = buscas[nome]["params"].iloc[0]
    print(f"{nome:20s} treinando... {params}", flush=True)
    t0 = time.time()
    base_m = construir(nome).set_params(**params).fit(Xtr, ytr)
    t_treino = time.time() - t0
    modelo = calibrar(base_m, Xcal, ycal) if CALIBRAR else base_m
    t0 = time.time()
    p = modelo.predict_proba(Xte)[:, 1]
    ms_1000 = (time.time() - t0) / len(yte) * 1e6
    p_sem = base_m.predict_proba(Xte)[:, 1]
    assert stats.spearmanr(p, p_sem).statistic > 0.99, f"{nome}: a calibração alterou a ordem dos scores"
    m = avaliar(yte, p, datas_te)
    m.update({
        "params": params,
        "pr_auc_cv": buscas[nome]["pr_auc_cv"].iloc[0],
        "brier_sem_calibracao": brier_score_loss(yte, p_sem),
        "limiar": limiar_f1(ycal, modelo.predict_proba(Xcal)[:, 1]),
        "segundos_treino": t_treino,
        "ms_por_1000_previsoes": ms_1000,
        "interpretabilidade": INTERPRETABILIDADE[nome],
    })
    try:
        imp_nome = importancia_nativa(modelo)
    except Exception as e:                         # a importância nunca deve derrubar o treino
        print(f"  importância de {nome} não calculada ({type(e).__name__}: {e})")
        imp_nome = pd.Series(dtype=float)
    finais[nome], res[nome], prev[nome], imp_nativa[nome] = modelo, m, p, imp_nome
    with open(arq, "wb") as f:                     # só métricas e previsões: arquivo pequeno e rápido
        pickle.dump((res[nome], prev[nome], imp_nativa[nome]), f)
    print(f"{nome:20s} pronto em {t_treino:.0f}s")

print()
for nome, m in res.items():
    print(f"{nome:20s} TESTE PR-AUC {m['pr_auc']:.4f} (IC95 {m['pr_auc_ic95'][0]:.4f}-{m['pr_auc_ic95'][1]:.4f}) | "
          f"ROC-AUC {m['roc_auc']:.4f} | Brier {m['brier_sem_calibracao']:.4f} -> {m['brier']:.4f} | "
          f"top10% {m['precisao_top10']:.1%}")

# Referência sem machine learning: taxa histórica de atraso da rota
p_ref = teste["taxa_atraso_rota_hist"].fillna(ytr.mean()).to_numpy()
ref = avaliar(yte, p_ref, datas_te)
print(f"\nReferência (taxa da rota) PR-AUC {ref['pr_auc']:.4f} | acaso = taxa base {ref['taxa_base']:.4f}")

In [0]:
# Matriz de critérios (pesos definidos na seção 0) e escolha do modelo
r = pd.DataFrame(res).T
for c in ["pr_auc", "pr_auc_cv", "brier", "precisao_top10", "variacao_mensal", "interpretabilidade",
          "ms_por_1000_previsoes", "segundos_treino"]:
    r[c] = r[c].astype(float)


def faixa(v, limites):
    return 1.0 if v <= limites[0] else 0.75 if v <= limites[1] else 0.5


queda = ((r["pr_auc_cv"] - r["pr_auc"]).abs() / r["pr_auc_cv"]).clip(0, 1)
matriz = pd.DataFrame({
    "discriminacao_pr_auc": r["pr_auc"] / r["pr_auc"].max(),
    "calibracao_brier": r["brier"].min() / r["brier"],
    "alto_risco_precisao_top10": r["precisao_top10"] / r["precisao_top10"].max(),
    "estabilidade": 1 - (queda + r["variacao_mensal"].clip(0, 1)) / 2,
    "interpretabilidade": r["interpretabilidade"],
    # latência por 1.000 previsões (<= 50 ms ótimo) e tempo de retreino (<= 1 min ótimo, <= 10 min aceitável)
    "viabilidade_operacional": [(faixa(a, (50, 500)) + faixa(b, (60, 600))) / 2
                                for a, b in zip(r["ms_por_1000_previsoes"], r["segundos_treino"])],
})
matriz["nota_final"] = sum(matriz[c] * w for c, w in PESOS_DECISAO.items())
matriz = matriz.sort_values("nota_final", ascending=False)

# Desempate: se a diferença de PR-AUC entre 1º e 2º pode ser ruído, vence o mais interpretável
vencedor, segundo = matriz.index[0], matriz.index[1]
rng = np.random.default_rng(SEED)
dif = []
for _ in range(300):
    i = rng.integers(0, len(yte), len(yte))
    if yte[i].sum() > 0:
        dif.append(average_precision_score(yte[i], prev[vencedor][i]) - average_precision_score(yte[i], prev[segundo][i]))
ic_dif = np.percentile(dif, [2.5, 97.5])
if ic_dif[0] <= 0 <= ic_dif[1] and INTERPRETABILIDADE[segundo] > INTERPRETABILIDADE[vencedor]:
    print(f"Desempate: {vencedor} e {segundo} empatam em PR-AUC (IC95 da diferença {ic_dif[0]:+.4f} a "
          f"{ic_dif[1]:+.4f}); vence o mais interpretável.")
    vencedor = segundo
else:
    print(f"Diferença de PR-AUC {vencedor} - {segundo}: IC95 {ic_dif[0]:+.4f} a {ic_dif[1]:+.4f}")

tabela = r[["pr_auc", "pr_auc_cv", "roc_auc", "brier_sem_calibracao", "brier", "precisao_top10", "lift_top10",
            "variacao_mensal", "limiar", "segundos_treino", "ms_por_1000_previsoes"]].astype(float)
tabela.loc["Referência (taxa da rota)", ["pr_auc", "roc_auc", "brier", "precisao_top10", "lift_top10"]] = \
    [ref["pr_auc"], ref["roc_auc"], ref["brier"], ref["precisao_top10"], ref["lift_top10"]]
print(f"Taxa base de atraso no teste (PR-AUC do acaso): {yte.mean():.4f}")
display(tabela.round(4).reset_index().rename(columns={"index": "modelo"}))
display(matriz.round(3).reset_index().rename(columns={"index": "modelo"}))
print(f"\n>>> MODELO ESCOLHIDO: {vencedor}")

Avaliação e diagnóstico

### Matrizes de confusão (Independente)


In [0]:
# Matrizes de confusão: os 4 modelos no TESTE (abr-ago/2018), em dois pontos de corte
#  1) Limiar F1: escolhido na calibração (jan-mar/2018), nunca no teste
#  2) Top 10% de maior risco: a "faixa vermelha" usada na matriz de decisão
# Usa prev, res, yte e MODELOS das células anteriores (rode a seção de treino final antes).

def metricas_corte(y, pred):
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"VN": vn, "FP": fp, "FN": fn, "VP": vp,
            "acuracia": (vp + vn) / len(y),
            "precisao": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "especificidade": vn / (vn + fp) if (vn + fp) else np.nan,
            "pct_sinalizados": pred.mean()}


CORTES = {
    "Limiar F1 (calibração)": lambda nome, p: (p >= res[nome]["limiar"]).astype(int),
    "Top 10% de risco": lambda nome, p: (p >= np.quantile(p, 0.9)).astype(int),
}

linhas = []
fig, axes = plt.subplots(len(CORTES), len(MODELOS), figsize=(4.3 * len(MODELOS), 4.1 * len(CORTES)))
for i, (corte, regra) in enumerate(CORTES.items()):
    for j, nome in enumerate(MODELOS):
        pred = regra(nome, prev[nome])
        m = metricas_corte(yte, pred)
        linhas.append({"corte": corte, "modelo": nome, **m, "pr_auc": float(res[nome]["pr_auc"])})
        cm = confusion_matrix(yte, pred, labels=[0, 1])
        anot = np.array([[f"{cm[a, b]:,}\n({cm[a, b] / cm[a].sum():.1%})" for b in range(2)] for a in range(2)])
        ax = axes[i, j]
        sns.heatmap(cm, annot=anot, fmt="", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=["No prazo", "Atrasou"], yticklabels=["No prazo", "Atrasou"])
        ax.set_title(f"{nome}\nprec {m['precisao']:.1%} | recall {m['recall']:.1%} | acur {m['acuracia']:.1%}",
                     fontsize=10)
        ax.set_xlabel("Previsto")
        ax.set_ylabel(f"{corte}\n\nReal" if j == 0 else "Real")
fig.suptitle("Matrizes de confusão no TESTE (abr-ago/2018) | % = parcela da classe real", y=1.02)
mostrar(fig, "matrizes_confusao")

# Referência ingênua: dizer que NENHUM pedido atrasa (mostra por que acurácia engana aqui)
linhas.append({"corte": "-", "modelo": "Ingênuo: 'nunca atrasa'",
               **metricas_corte(yte, np.zeros_like(yte)), "pr_auc": yte.mean()})

comparacao = pd.DataFrame(linhas)
cols_pct = ["acuracia", "precisao", "recall", "f1", "especificidade", "pct_sinalizados"]
display(comparacao.style.format({c: "{:.1%}" for c in cols_pct} | {"pr_auc": "{:.4f}"}))

# Diferença de PR-AUC: modelo escolhido contra cada um dos outros (bootstrap pareado no teste)
escolhido = vencedor if "vencedor" in globals() else "Regressão Logística"
rng = np.random.default_rng(SEED)
idx_boot = [i for i in (rng.integers(0, len(yte), len(yte)) for _ in range(300)) if yte[i].sum() > 0]
print(f"\nDiferença de PR-AUC ({escolhido} menos o outro), IC95 por bootstrap pareado:")
for nome in MODELOS:
    if nome == escolhido:
        continue
    dif = [average_precision_score(yte[i], prev[escolhido][i]) - average_precision_score(yte[i], prev[nome][i])
           for i in idx_boot]
    lo, hi = np.percentile(dif, [2.5, 97.5])
    veredito = "diferença real" if lo > 0 or hi < 0 else "empate estatístico"
    print(f"  vs {nome:18s} {res[escolhido]['pr_auc'] - res[nome]['pr_auc']:+.4f} "
          f"(IC95 {lo:+.4f} a {hi:+.4f}) -> {veredito}")

In [0]:
# Curva Precision-Recall no TESTE: 4 modelos + acaso + referência da rota
# Os marcadores mostram os dois cortes da matriz de confusão (limiar F1 e top 10%).
base_te = yte.mean()
cores = dict(zip(MODELOS, sns.color_palette("deep", len(MODELOS))))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
ax = axes[0]
for nome in MODELOS:
    p = prev[nome]
    pr, rc, _ = precision_recall_curve(yte, p)
    ax.plot(rc, pr, color=cores[nome], lw=1.8, label=f"{nome} (PR-AUC {res[nome]['pr_auc']:.3f})")
    for regra, marcador in [(p >= res[nome]["limiar"], "o"), (p >= np.quantile(p, 0.9), "s")]:
        ax.scatter(recall_score(yte, regra), precision_score(yte, regra, zero_division=0),
                   color=cores[nome], marker=marcador, s=60, edgecolor="black", zorder=5)
pr_r, rc_r, _ = precision_recall_curve(yte, p_ref)
ax.plot(rc_r, pr_r, color="gray", ls="--", lw=1.2, label=f"Referência: taxa da rota ({ref['pr_auc']:.3f})")
ax.axhline(base_te, color="red", ls=":", lw=1.5, label=f"Acaso = taxa base ({base_te:.3f})")
ax.scatter([], [], marker="o", color="white", edgecolor="black", label="Limiar F1 (calibração)")
ax.scatter([], [], marker="s", color="white", edgecolor="black", label="Top 10% de risco")
ax.set(xlabel="Recall (atrasos capturados)", ylabel="Precisão (alertas corretos)",
       title="Curva Precision-Recall | teste abr-ago/2018", xlim=(0, 1), ylim=(0, 0.5))
ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
ax.legend(fontsize=8, loc="upper right")

# Precisão por faixa de pedidos sinalizados (visão operacional: "se eu checar os X% mais arriscados...")
ax = axes[1]
fracoes = np.linspace(0.01, 0.5, 50)
for nome in MODELOS:
    ordem = np.argsort(-prev[nome])
    ax.plot(fracoes, [yte[ordem[:max(1, int(f * len(yte)))]].mean() for f in fracoes],
            color=cores[nome], lw=1.8, label=nome)
ax.axhline(base_te, color="red", ls=":", lw=1.5, label="Acaso")
ax.axvline(0.10, color="black", ls="--", lw=0.8)
ax.set(xlabel="% de pedidos sinalizados (maior risco primeiro)", ylabel="Precisão na faixa",
       title="Precisão por tamanho da faixa de risco")
ax.xaxis.set_major_formatter(PCT); ax.yaxis.set_major_formatter(PCT)
ax.legend(fontsize=8)
mostrar(fig, "curva_precision_recall")

# Precisão em níveis fixos de recall (melhor precisão alcançável com pelo menos aquele recall)
linhas_pr = []
for nome in MODELOS:
    pr, rc, _ = precision_recall_curve(yte, prev[nome])
    linha = {"modelo": nome, "pr_auc": res[nome]["pr_auc"], "lift_pr_auc": res[nome]["pr_auc"] / base_te}
    for alvo in (0.1, 0.2, 0.3, 0.5, 0.7):
        linha[f"precisao@recall{int(alvo * 100)}"] = pr[rc >= alvo].max()
    linhas_pr.append(linha)
tab_pr = pd.DataFrame(linhas_pr).set_index("modelo")
display(tab_pr.style.format({c: "{:.1%}" for c in tab_pr.columns if c.startswith("precisao")}
                            | {"pr_auc": "{:.4f}", "lift_pr_auc": "{:.2f}x"}))